## Nested Loops Insights + Arithmetic Sequence

In [ ]:
def buy_sell_stock_brute(prices: list[int]) -> int:
    max_profit = 0

    for i in range(len(prices) - 1):
        for j in range(i + 1, len(prices)):
            if prices[j] > prices[i]:
                max_profit = max(prices[j] - prices[i], max_profit)

    return max_profit

For `n` items we check `n - i - 1`.

**1. Why `n-i` per iteration is still $O(n²)$:** You're right that iteration `i` only checks `n - i - 1` items, not `n`. But the _total_ work across all iterations is:

(n-1) + (n-2) + (n-3) + ... + 1 + 0

That's the sum of an **arithmetic sequence**, and it has a closed form: `n(n-1)/2`. Expand that: `(n² - n)/2`. When you drop constants and lower-order terms (the `-n` and the `/2`), what's left is `n²`. So even though each individual iteration does less work than the last, the **sum** of a linearly-shrinking sequence is still quadratic — this is worth filing permanently, you'll see this exact sum in 3Sum, bubble sort, and anywhere else you have a triangular nested-loop shape.

**#2 — the real, unconditional reason nested loops are $O(n^2)$ — just in terms of what the two loops do:** it's structural, not data-dependent.
> The nested loop always performs `n(n-1)/2` comparisons for an array of size `n` — that count is fixed by the loop _shape_ alone. It doesn't matter what the values are, whether there are duplicates, or whether you add early exits that don't actually fire in the worst case.
That's why "duplicates" and "no stop-early" were the wrong diagnosis: neither one is _the_ reason — the reason is **two nested loops over the same collection**, full stop.

**#3 — the redundant computation, revealed:** For a fixed sell day `j`, the best possible profit is `prices[j] - (the smallest price among all days before j)`.
> In your brute force, buying day `i` and selling day `j` **recomputes "what's the smallest price so far"** implicitly and separately for every `i` — the outer loop **re-derives context from scratch** each time instead of carrying it forward.

The single value worth **carrying forward** as you scan left to right, once, is: **the minimum price seen so far**. At each day `j`, compute `profit = prices[j] - min_so_far`, track the best profit seen, then update `min_so_far = min(min_so_far, prices[j])` for the next day. One pass, one running variable — no inner loop at all.

In [ ]:
def buy_sell_stock(prices: list[int]) -> int:
    left = 0
    max_profit = 0

    for right in range(1, len(prices)):
        if prices[right] > prices[left]:
            profit = prices[right] - prices[left]
            max_profit = max(max_profit, profit)
        else:
            left += 1
    return max_profit

**Notice**: `left` never goes backward, and `prices[left]` **is** "the minimum price seen so far" — you don't need a separate `min_so_far` variable at all, `left` already points at it. That's the whole insight: one forward pass, `right` scans, `left` only jumps forward when a new low appears.

> **Key insight**: `left` always points at the **minimum price seen so far**, so no separate scan or variable is needed to track it — one forward pass compares every day against the best buy day found up to that point.

### Arithmetic Sequence
- **`aₙ = a₁ + (n-1)d`** — gives you a **single term**, the value at position `n`. Not what we needed here.
- **`Sₙ = n/2 × (a₁ + aₙ)`** — gives you the **sum** of the first `n` terms. This is the one we needed, since we were adding up `(n-1) + (n-2) + ... + 1 + 0`.

Plug into the sum formula: `a₁ = 0` (the last, smallest term), `aₙ = n-1` (the first, biggest term), and there are `n` terms total. `Sₙ = n/2 × (0 + (n-1)) = n(n-1)/2` — that's exactly where it came from. The trick to notice: `Sₙ` is derived by pairing the first and last term, second and second-last, etc. — each pair sums to the same value `(a₁ + aₙ)`, and there are `n/2` such pairs.